# ProteinOPD Inference

Generate protein sequences with the released ProteinOPD preference-aligned adapters.

Use a GPU runtime: `Runtime > Change runtime type > T4/L4/A100 GPU`.

If the base model or adapter is private or gated, run the Hugging Face login cell before generation.

In [ ]:
#@title Setup
repo_url = "https://github.com/THU-AI4S/ProteinOPD.git" #@param {type:"string"}
branch = "main" #@param {type:"string"}

import os
import subprocess
import sys
from pathlib import Path

def run(cmd, cwd=None):
    print("$", " ".join(str(x) for x in cmd))
    result = subprocess.run(cmd, cwd=cwd, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    if result.stdout:
        print(result.stdout)
    result.check_returncode()

run(["nvidia-smi"])

repo_dir = Path("/content/ProteinOPD")
if repo_dir.exists():
    run(["git", "fetch", "origin"], cwd=repo_dir)
    run(["git", "checkout", branch], cwd=repo_dir)
    run(["git", "pull", "--ff-only"], cwd=repo_dir)
else:
    run(["git", "clone", "--branch", branch, repo_url, str(repo_dir)])

run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"])

PINNED_PACKAGES = [
    "accelerate==1.12.0",
    "datasets==4.4.2",
    "huggingface_hub",
    "peft==0.18.0",
    "pyyaml",
    "safetensors==0.7.0",
    "sentencepiece==0.2.1",
    "tokenizers==0.22.2",
    "tqdm",
    "transformers==4.57.3",
]
run([sys.executable, "-m", "pip", "install", "-q", *PINNED_PACKAGES])

import importlib.metadata as md
for pkg in ["torch", "transformers", "peft", "accelerate", "datasets", "huggingface_hub", "tokenizers", "safetensors", "sentencepiece", "torchao"]:
    try:
        print(f"{pkg}=={md.version(pkg)}")
    except md.PackageNotFoundError:
        print(f"{pkg} not installed")

print("Repository:", repo_dir)

In [ ]:
#@title Optional Hugging Face login
login_to_huggingface = False #@param {type:"boolean"}

if login_to_huggingface:
    from huggingface_hub import notebook_login
    notebook_login()
else:
    print("Skipping Hugging Face login. Enable this for private or gated models.")

In [ ]:
#@title Generation settings
model_type = "unconditional" #@param ["unconditional", "conditional"]

unconditional_base_model_path_or_id = "nferruz/ProtGPT2" #@param {type:"string"}
unconditional_adapter_path_or_id = "purilion/ProteinOPD/protgpt2-adapter" #@param {type:"string"}
conditional_base_model_path_or_id = "GreatCaptainNemo/ProLLaMA" #@param {type:"string"}
conditional_adapter_path_or_id = "purilion/ProteinOPD/prollama-adapter" #@param {type:"string"}

num_sequences = 5 #@param {type:"integer"}
batch_size = 2 #@param {type:"integer"}
max_new_tokens = 256 #@param {type:"integer"}
temperature = 0.7 #@param {type:"number"}
top_p = 0.9 #@param {type:"number"}
top_k = 200 #@param {type:"integer"}
repetition_penalty = 1.2 #@param {type:"number"}
seed = 42 #@param {type:"integer"}

superfamily = "Lysozyme-like domain superfamily" #@param {type:"string"}
seq_prefix = "" #@param {type:"string"}

base_model_path_or_id = unconditional_base_model_path_or_id if model_type == "unconditional" else conditional_base_model_path_or_id
adapter_path_or_id = unconditional_adapter_path_or_id if model_type == "unconditional" else conditional_adapter_path_or_id

output_dir = Path("/content/proteinopd_outputs")
output_dir.mkdir(parents=True, exist_ok=True)
json_path = output_dir / f"{model_type}_generated.json"
fasta_path = output_dir / f"{model_type}_generated.fasta"

print("Base model:", base_model_path_or_id)
print("Adapter:", adapter_path_or_id)
print("JSON output:", json_path)
print("FASTA output:", fasta_path)

In [ ]:
import subprocess
import sys
from pathlib import Path
from huggingface_hub import snapshot_download


def resolve_hf_subfolder_or_path(path_or_id):
    candidate = Path(path_or_id).expanduser()
    if candidate.exists():
        return str(candidate.resolve())
    parts = path_or_id.strip("/").split("/")
    if len(parts) <= 2:
        return path_or_id
    repo_id = "/".join(parts[:2])
    subfolder = "/".join(parts[2:])
    print(f"Downloading Hugging Face subfolder: repo_id={repo_id}, subfolder={subfolder}")
    snapshot_root = Path(snapshot_download(repo_id=repo_id, allow_patterns=[f"{subfolder}/*"]))
    resolved = snapshot_root / subfolder
    if not resolved.exists():
        raise FileNotFoundError(f"Adapter subfolder was not found after download: {resolved}")
    adapter_config = resolved / "adapter_config.json"
    if not adapter_config.is_file():
        raise FileNotFoundError(f"Adapter config not found at {adapter_config}. Check that {path_or_id} points to a PEFT adapter directory.")
    return str(resolved)


resolved_adapter_path = resolve_hf_subfolder_or_path(adapter_path_or_id)

if model_type == "unconditional":
    cmd = [
        sys.executable, "unconditional/generate/generate.py",
        "--base_model_path", base_model_path_or_id,
        "--adapter_path", resolved_adapter_path,
        "--output_json_path", str(json_path),
        "--num_sequences", str(num_sequences),
        "--batch_size", str(batch_size),
        "--max_new_tokens", str(max_new_tokens),
        "--temperature", str(temperature),
        "--top_p", str(top_p),
        "--top_k", str(top_k),
        "--repetition_penalty", str(repetition_penalty),
        "--seed", str(seed),
    ]
else:
    cmd = [
        sys.executable, "conditional/generate/generate.py",
        "--load_mode", "lora",
        "--model", base_model_path_or_id,
        "--adapter_path", resolved_adapter_path,
        "--superfamily", superfamily,
        "--num_sequences", str(num_sequences),
        "--generation_batch_size", str(batch_size),
        "--output_json", str(json_path),
        "--temperature", str(temperature),
        "--top_p", str(top_p),
        "--top_k", str(top_k),
        "--repetition_penalty", str(repetition_penalty),
        "--max_new_tokens", str(max_new_tokens),
        "--seed", str(seed),
    ]
    if seq_prefix.strip():
        cmd.extend(["--seq_prefix", seq_prefix.strip()])

print("Resolved adapter path:", resolved_adapter_path)
run(cmd, cwd=repo_dir)

In [ ]:
import json
import pandas as pd

records = json.loads(json_path.read_text())

with fasta_path.open("w", encoding="utf-8") as handle:
    for idx, record in enumerate(records, start=1):
        sequence = record.get("response#1", "")
        handle.write(f">ProteinOPD_{model_type}_{idx}\n{sequence}\n")

df = pd.DataFrame(records)
display(df.head(20))
print("Saved JSON:", json_path)
print("Saved FASTA:", fasta_path)